In [0]:
%pip install dbldatagen

In [0]:
%restart_python

In [0]:
base_path = "/Volumes/capstone_project/capstone_schema/capstone_volume"

In [0]:
import dbldatagen as dg
from pyspark.sql.functions import rand, when, col
from datetime import timedelta, datetime

# --- 1. Parameters ---
BASE_ROW_COUNT = 1900000 
NUM_DAYS = 90

end_date = datetime.now()
start_date = end_date - timedelta(days=NUM_DAYS)

# --- 2. Build the Schema & Inject Flaws using dbldatagen ---
data_spec = (
    dg.DataGenerator(spark, name="ecommerce_orders_raw", rows=BASE_ROW_COUNT, partitions=8)
    
    # FLAW 1 (Null Handling): Inject 5% NULL values directly into order_id
    .withColumn("order_id", "string", 
                expr="concat('ORD-', lpad(cast(int(rand() * 99999999) as string), 8, '0'))", 
                percentNulls=0.05)
    
    # THE FIX: Bounded pools for realistic repeat purchases (15k customers, 500 products)
    .withColumn("customer_id", "string", 
                expr="concat('CUST-', lpad(cast(int(rand() * 15000) as string), 5, '0'))")
    .withColumn("product_id", "string", 
                expr="concat('PROD-', lpad(cast(int(rand() * 500) as string), 4, '0'))")
    
    # FLAW 2 (Category Normalization): Messy free-text categories
    .withColumn("category", "string", 
                values=["Electronics", "ELECTRONICS", "electronics ", "Elect.", 
                        "Apparel", "apparel", "APP", 
                        "Beauty", "beauty products",
                        "sports", "Sport", "Sports"])
    
    # FIXED: Added quantity back in
    .withColumn("quantity", "integer", minValue=-5, maxValue=10, random=True) 
    
    # FIXED: Placed unit_price on its own line
    .withColumn("unit_price", "float", expr="(rand() * 1550) - 50") 
    
    # FIXED: Only ONE discount column, using the high-variance expr method
    .withColumn("discount", "float", expr="rand() * 0.4", percentNulls=0.02)
    
    .withColumn("payment_method", "string", values=["Credit Card", "UPI", "Debit Card"], random=True)
    .withColumn("device_type", "string", values=["Mobile", "Desktop", "Tablet"], random=True)
    # FIXED: Using rock-solid Unix timestamp math to avoid Spark interval parsing errors.
    # 90 days * 24 hours * 60 minutes * 60 seconds = absolute uniform distribution!
    .withColumn("event_timestamp", "timestamp",
                expr=f"cast(from_unixtime(unix_timestamp('{start_date.strftime('%Y-%m-%d 00:00:00')}') + cast(rand() * {NUM_DAYS * 24 * 60 * 60} as bigint)) as timestamp)")
)

# Build the base DataFrame
print(f"Building base {BASE_ROW_COUNT} rows...")
df_base = data_spec.build()

# --- 3. FLAW 4 (Deduplication): Create Exact Duplicates ---
print("Injecting duplicate records for MERGE INTO testing...")
df_duplicates = df_base.sample(withReplacement=False, fraction=0.05)
df_dirty_final = df_base.union(df_duplicates)

# --- 4. Write to Cloud Storage ---
print(f"Writing 5 Million rows of messy data to {base_path}/input in JSON format...")
df_dirty_final.write.format("json").mode("overwrite").save(f"{base_path}/input")

print("Data generation complete! Your Bronze layer is ready for ingestion.")
display(df_dirty_final)

In [0]:
dataset = spark.read.format('json').load(f'{base_path}/input')

In [0]:
dataset.count()

In [0]:
dataset.printSchema()

In [0]:
from pyspark.sql.functions import col
dataset.filter(col("unit_price") < 0).count()

In [0]:
from pyspark.sql.functions import col
dataset.select(col('customer_id')).distinct().count()

In [0]:
from pyspark.sql import functions as F
dataset.select(F.max('customer_id'), F.min('customer_id')).show()

In [0]:
from pyspark.sql import functions as F

In [0]:
dataset.filter(F.col('discount') != 0).count()

### 2 Million New data

In [0]:
import dbldatagen as dg
from pyspark.sql.functions import rand, when, col
from datetime import timedelta, datetime

# --- 1. Parameters ---
BASE_ROW_COUNT = 1900000 
NUM_DAYS = 90

end_date = datetime.now()
start_date = end_date - timedelta(days=NUM_DAYS)

# --- 2. Build the Schema & Inject Flaws using dbldatagen ---
data_spec = (
    dg.DataGenerator(spark, name="ecommerce_orders_raw", rows=BASE_ROW_COUNT, partitions=8)
    
    # FLAW 1 (Null Handling): Inject 5% NULL values directly into order_id
    .withColumn("order_id", "string", 
                expr="concat('ORD-', lpad(cast(int(rand() * 99999999) as string), 8, '0'))", 
                percentNulls=0.05)
    
    # THE FIX: Bounded pools for realistic repeat purchases (15k customers, 500 products)
    .withColumn("customer_id", "string", 
                expr="concat('CUST-', lpad(cast(int(rand() * 15000) as string), 5, '0'))")
    .withColumn("product_id", "string", 
                expr="concat('PROD-', lpad(cast(int(rand() * 500) as string), 4, '0'))")
    
    # FLAW 2 (Category Normalization): Messy free-text categories
    .withColumn("category", "string", 
                values=["Electronics", "ELECTRONICS", "electronics ", "Elect.", 
                        "Apparel", "apparel", "APP", 
                        "Beauty", "beauty products",
                        "sports", "Sport", "Sports"])
    
    # FIXED: Added quantity back in
    .withColumn("quantity", "integer", minValue=-5, maxValue=10, random=True) 
    
    # FIXED: Placed unit_price on its own line
    .withColumn("unit_price", "float", expr="(rand() * 1550) - 50") 
    
    # FIXED: Only ONE discount column, using the high-variance expr method
    .withColumn("discount", "float", expr="rand() * 0.4", percentNulls=0.02)
    
    .withColumn("payment_method", "string", values=["Credit Card", "UPI", "Debit Card"], random=True)
    .withColumn("device_type", "string", values=["Mobile", "Desktop", "Tablet"], random=True)
    # FIXED: Using rock-solid Unix timestamp math to avoid Spark interval parsing errors.
    # 90 days * 24 hours * 60 minutes * 60 seconds = absolute uniform distribution!
    .withColumn("event_timestamp", "timestamp",
                expr=f"cast(from_unixtime(unix_timestamp('{start_date.strftime('%Y-%m-%d 00:00:00')}') + cast(rand() * {NUM_DAYS * 24 * 60 * 60} as bigint)) as timestamp)")
)

# Build the base DataFrame
print(f"Building base {BASE_ROW_COUNT} rows...")
df_base = data_spec.build()

# --- 3. FLAW 4 (Deduplication): Create Exact Duplicates ---
print("Injecting duplicate records for MERGE INTO testing...")
df_duplicates = df_base.sample(withReplacement=False, fraction=0.05)
df_dirty_final = df_base.union(df_duplicates)

# --- 4. Write to Cloud Storage ---
print(f"Writing 5 Million rows of messy data to {base_path}/input in JSON format...")
df_dirty_final.write.format("json").mode("append").save(f"{base_path}/input")

print("Data generation complete! Your Bronze layer is ready for ingestion.")
display(df_dirty_final)

In [0]:
dataset = spark.read.format('json').load(f'{base_path}/input')
dataset.count()

In [0]:
data = [
    {"category": "Electronics", "customer_id": "CUST-15000", "device_type": "Mobile", "discount": 0.12, "event_timestamp": "2026-04-10T08:15:00.000Z", "order_id": "ORD-84729101", "payment_method": "Credit Card", "product_id": "PROD-0105", "quantity": 1, "unit_price": 299.99},
    {"category": "Apparel", "customer_id": "CUST-15001", "device_type": "Desktop", "discount": 0.05, "event_timestamp": "2026-04-10T09:20:12.000Z", "order_id": "ORD-84729102", "payment_method": "UPI", "product_id": "PROD-0210", "quantity": 2, "unit_price": 45.50},
    {"category": "Home", "customer_id": "CUST-15002", "device_type": "Tablet", "discount": 0.20, "event_timestamp": "2026-04-10T09:45:33.000Z", "order_id": "ORD-84729103", "payment_method": "Debit Card", "product_id": "PROD-0302", "quantity": 1, "unit_price": 150.00},
    {"category": "Sports", "customer_id": "CUST-15003", "device_type": "Mobile", "discount": 0.0, "event_timestamp": "2026-04-10T10:11:05.000Z", "order_id": "ORD-84729104", "payment_method": "Credit Card", "product_id": "PROD-0415", "quantity": 3, "unit_price": 25.99},
    {"category": "Electronics", "customer_id": "CUST-15004", "device_type": "Desktop", "discount": 0.10, "event_timestamp": "2026-04-10T11:05:42.000Z", "order_id": "ORD-84729105", "payment_method": "UPI", "product_id": "PROD-0112", "quantity": 1, "unit_price": 899.50},
    {"category": "Apparel", "customer_id": "CUST-15005", "device_type": "Mobile", "discount": 0.15, "event_timestamp": "2026-04-10T11:30:19.000Z", "order_id": "ORD-84729106", "payment_method": "Wallet", "product_id": "PROD-0245", "quantity": 4, "unit_price": 19.99},
    {"category": "Home", "customer_id": "CUST-15006", "device_type": "Desktop", "discount": 0.0, "event_timestamp": "2026-04-10T12:00:00.000Z", "order_id": "ORD-84729107", "payment_method": "Credit Card", "product_id": "PROD-0350", "quantity": 2, "unit_price": 75.25},
    {"category": "Sports", "customer_id": "CUST-15007", "device_type": "Tablet", "discount": 0.05, "event_timestamp": "2026-04-10T13:15:22.000Z", "order_id": "ORD-84729108", "payment_method": "Debit Card", "product_id": "PROD-0480", "quantity": 1, "unit_price": 120.00},
    {"category": "Electronics", "customer_id": "CUST-15008", "device_type": "Mobile", "discount": 0.25, "event_timestamp": "2026-04-10T14:22:11.000Z", "order_id": "ORD-84729109", "payment_method": "UPI", "product_id": "PROD-0199", "quantity": 1, "unit_price": 1200.00},
    {"category": "Apparel", "customer_id": "CUST-15009", "device_type": "Desktop", "discount": 0.10, "event_timestamp": "2026-04-10T14:50:35.000Z", "order_id": "ORD-84729110", "payment_method": "Credit Card", "product_id": "PROD-0276", "quantity": 2, "unit_price": 55.40},
    {"category": "Home", "customer_id": "CUST-15010", "device_type": "Mobile", "discount": 0.0, "event_timestamp": "2026-04-10T15:10:05.000Z", "order_id": "ORD-84729111", "payment_method": "Wallet", "product_id": "PROD-0311", "quantity": 5, "unit_price": 12.50},
    {"category": "Sports", "customer_id": "CUST-15011", "device_type": "Desktop", "discount": 0.15, "event_timestamp": "2026-04-10T15:45:50.000Z", "order_id": "ORD-84729112", "payment_method": "Debit Card", "product_id": "PROD-0432", "quantity": 1, "unit_price": 210.99},
    {"category": "Electronics", "customer_id": "CUST-15012", "device_type": "Tablet", "discount": 0.05, "event_timestamp": "2026-04-10T16:05:12.000Z", "order_id": "ORD-84729113", "payment_method": "Credit Card", "product_id": "PROD-0144", "quantity": 2, "unit_price": 350.00},
    {"category": "Apparel", "customer_id": "CUST-15013", "device_type": "Mobile", "discount": 0.20, "event_timestamp": "2026-04-10T16:30:25.000Z", "order_id": "ORD-84729114", "payment_method": "UPI", "product_id": "PROD-0291", "quantity": 3, "unit_price": 30.00},
    {"category": "Home", "customer_id": "CUST-15014", "device_type": "Desktop", "discount": 0.10, "event_timestamp": "2026-04-10T17:15:40.000Z", "order_id": "ORD-84729115", "payment_method": "Credit Card", "product_id": "PROD-0388", "quantity": 1, "unit_price": 450.75},
    {"category": "Sports", "customer_id": "CUST-15015", "device_type": "Mobile", "discount": 0.0, "event_timestamp": "2026-04-10T17:50:55.000Z", "order_id": "ORD-84729116", "payment_method": "Debit Card", "product_id": "PROD-0477", "quantity": 2, "unit_price": 85.20},
    {"category": "Electronics", "customer_id": "CUST-15016", "device_type": "Tablet", "discount": 0.12, "event_timestamp": "2026-04-10T18:20:10.000Z", "order_id": "ORD-84729117", "payment_method": "UPI", "product_id": "PROD-0165", "quantity": 1, "unit_price": 150.00},
    {"category": "Apparel", "customer_id": "CUST-15017", "device_type": "Desktop", "discount": 0.08, "event_timestamp": "2026-04-10T19:05:30.000Z", "order_id": "ORD-84729118", "payment_method": "Credit Card", "product_id": "PROD-0222", "quantity": 4, "unit_price": 22.50},
    {"category": "Home", "customer_id": "CUST-15018", "device_type": "Mobile", "discount": 0.18, "event_timestamp": "2026-04-10T19:45:45.000Z", "order_id": "ORD-84729119", "payment_method": "Wallet", "product_id": "PROD-0344", "quantity": 1, "unit_price": 55.00},
    {"category": "Sports", "customer_id": "CUST-15019", "device_type": "Desktop", "discount": 0.05, "event_timestamp": "2026-04-10T20:10:00.000Z", "order_id": "ORD-84729120", "payment_method": "Credit Card", "product_id": "PROD-0499", "quantity": 2, "unit_price": 145.99}
]

# Create the DataFrame
df_dirty_final = spark.createDataFrame(data)

# Use "append" to add these 20 new rows to your existing 5 rows!
df_dirty_final.write.format("json").mode("append").save(f"{base_path}/input")

In [0]:
from pyspark.sql import functions as F
df = spark.read.format('json').load(f"{base_path}/input")
df.count()
df.select(F.max('customer_id'), F.min('customer_id')).show()

In [0]:
data = [
    {"category": "Electronics", "customer_id": "CUST-15000", "device_type": "Mobile", "discount": 0.12, "event_timestamp": "2026-04-10T08:15:00.000Z", "order_id": "ORD-84729101", "payment_method": "Credit Card", "product_id": "PROD-0105", "quantity": 1, "unit_price": 299.99}
]

In [0]:
# Create the DataFrame
df_dirty_final = spark.createDataFrame(data)

# Use "append" to add these 20 new rows to your existing 5 rows!
df_dirty_final.write.format("json").mode("append").save(f"{base_path}/input")

In [0]:
from pyspark.sql import functions as F
df = spark.read.format('json').load(f"{base_path}/input")
df.count()
df.select(F.max('customer_id'), F.min('customer_id')).show()

In [0]:
from pyspark.sql import functions as F
base_path = "/Volumes/capstone_project/capstone_schema/capstone_volume"
df = spark.read.format('json').load(f"{base_path}/input")
print(df.count())